# Diacritics and the ASCII Trap

**Computational Text Analysis Pack · Module 03 — Preparing Text**

*Technique:* Unicode-aware tokenising; measuring what naive cleaning destroys

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/adh-dh-notebooks/blob/main/01-preparing-text/03b_diacritics_and_the_ascii_trap.ipynb)

Data: `masakhanews.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import unicodedata
from collections import Counter

# The data this notebook needs. It is downloaded from CourseHub automatically;
# uploading by hand is only a fallback if that server is unreachable.
NEEDED = ['masakhanews.csv']
DATA_URL = 'https://coursehub.africandigitalhumanities.ca/ide/pack-data/text/'

def _missing():
    return [f for f in NEEDED if not os.path.exists(f)]

missing = _missing()
if missing:
    # Fetch from CourseHub first, so nobody has to upload anything.
    import urllib.request, urllib.parse
    for _f in list(missing):
        try:
            urllib.request.urlretrieve(DATA_URL + urllib.parse.quote(_f), _f)
            print(f'downloaded {_f}')
        except Exception as _e:
            print(f'could not fetch {_f}: {_e}')
    missing = _missing()

if missing:
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))
    # Ask again until everything has arrived. The upload widget returns as soon
    # as you close it, so picking only some of the files would otherwise fail a
    # few lines below with a confusing FileNotFoundError.
    for _ in range(4):
        print('Select ALL of these at once (ctrl-click / cmd-click to multi-select):')
        print('   ' + ', '.join(missing))
        files.upload()
        missing = _missing()
        if not missing:
            break
        print('Still needed: ' + ', '.join(missing))
    if missing:
        raise SystemExit(
            'Missing: ' + ', '.join(missing) + '. Re-run this cell and select '
            'every file listed, or upload them with the folder icon on the left.')

df       = pd.read_csv('masakhanews.csv')
NIGERIAN = ['Hausa', 'Igbo', 'Yoruba', 'Nigerian Pidgin']
# Most of this pack runs on scikit-learn and the standard library. Also
# available: nltk, spacy, gensim and networkx (modules 14-19), Keras
# (case studies), and a GPU pod reached through /ide/api/neural/ for the
# neural modules — never called directly, so no token lives in a notebook.

plt.rcParams['figure.figsize'] = (10, 5.5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25

print(f'Loaded. df has {len(df):,} rows and {df.shape[1]} columns.')

## What this notebook does

The commonest "clean your text" line on the internet is some version of:

```python
text = re.sub(r"[^a-z ]", "", text.lower())
```

It is in tutorials, in textbooks, in published code. It also silently deletes
Yoruba tone marks, Yoruba sub-dot vowels, and the Hausa hooked consonants ƙ ɗ ɓ ƴ
— which are not decoration. They are separate letters that change meaning.

This notebook counts exactly how many characters that line destroys, and shows
you the words it turns into different words.

## The tooling

**`re`** with Unicode-aware `\w`, which is what keeps Yorùbá and Hausa
characters alive. **NLTK** and **spaCy** ship tokenisers that are faster and
smarter for English and frequently wrong for African orthographies — module 14
measures that directly.

In [ ]:
import re

def tokenise(text):
    """Words, lowercased, Unicode-aware.

    `\\w+` matches Unicode letters, so ọ, à, ƙ, ɗ and ɓ survive. The common
    shortcut `[a-z]+` would throw them away and quietly change every count in
    this notebook — see notebook 03b.
    """
    return re.findall(r"\w+", str(text).lower())

sample = "Àwọn ọmọ ìlú ti ṣe ìpàdé — ƙungiyar ɗalibai ta yi taro"
print("sample     :", sample)
print("tokens     :", tokenise(sample))
print("naive [a-z]:", re.findall(r"[a-z]+", sample.lower()))
print()
print("The naive pattern splits words at every accented letter.")

## The characters at risk

Yoruba uses ọ ṣ ẹ (a dot below) and three tones (à á, è é, and so on). Hausa uses
ƙ ɗ ɓ ƴ — hooked letters representing ejective and implosive consonants that
English has no equivalent for.

In [ ]:
AT_RISK = "ọṣẹàáèéìíòóùúńǹẹ́ọ́ẹ̀ọ̀ƙɗɓƴƙ̀"
sample = df[df["language"].isin(["Yoruba", "Hausa", "Igbo"])]

by_lang = {}
for lang, g in sample.groupby("language"):
    text = " ".join(str(t) for t in g["text"])
    c = Counter(ch for ch in text if ch.lower() in AT_RISK)
    by_lang[lang] = (len(text), c)
    print(f"{lang}: {len(text):,} characters")
    for ch, n in c.most_common(10):
        name = unicodedata.name(ch, "?")
        print(f"    {ch}   {n:>7,}   {name}")
    print()

## What the naive line destroys

Now run the tutorial line and count the difference. This is not a hypothetical:
it is the exact code people paste.

In [ ]:
import re

def naive_clean(text):
    # The line from a thousand tutorials.
    return re.sub(r"[^a-z ]", " ", str(text).lower())

LANGUAGE = "Yoruba"         # ← change this

text = " ".join(str(t) for t in df[df["language"] == LANGUAGE]["text"])
cleaned = naive_clean(text)

before = Counter(ch for ch in text.lower() if ch.isalpha())
after = Counter(ch for ch in cleaned if ch.isalpha())
lost = {ch: before[ch] - after.get(ch, 0) for ch in before}
lost = {ch: n for ch, n in lost.items() if n > 0}

print(f"{LANGUAGE}: {len(text):,} characters in the corpus")
print(f"letters destroyed by the naive clean: {sum(lost.values()):,}")
print()
print(f"{'char':<6}{'lost':>9}   name")
print("-" * 56)
for ch, n in sorted(lost.items(), key=lambda kv: -kv[1])[:12]:
    print(f"{ch:<6}{n:>9,}   {unicodedata.name(ch, '?')}")

## The damage in words, not characters

Counts are abstract. Look at what happens to actual words.

In [ ]:
counts = Counter(tokenise(text))
examples = [w for w, _ in counts.most_common(400)
            if any(ch in AT_RISK for ch in w)][:12]

print(f"{'original':<20}{'after naive clean':<24}same?")
print("-" * 54)
for w in examples:
    mangled = re.sub(r"[^a-z]", "", w.lower())
    print(f"{w:<20}{mangled or '(deleted entirely)':<24}"
          f"{'yes' if mangled == w else 'NO'}")

In [ ]:
# Two words that differ only by a diacritic are now the same word.
# In Yoruba tone is lexical: change the tone, change the meaning.
pairs = [("ọkọ", "husband / vehicle — the sub-dot is not optional"),
         ("owó", "money"),
         ("owo", "broom / hand, depending on tone"),
         ("ìlú", "town"),
         ("ilu", "drum")]
print("Words the naive clean collapses together:")
seen = {}
for w, gloss in pairs:
    flat = re.sub(r"[^a-z]", "", unicodedata.normalize("NFD", w).lower())
    seen.setdefault(flat, []).append(f"{w} ({gloss})")
for flat, group in seen.items():
    if len(group) > 1:
        print(f"\n  all become '{flat}':")
        for g in group:
            print(f"     {g}")
print()
print("A frequency count on naively cleaned Yoruba merges distinct words.")
print("No error is raised. The numbers simply mean something else.")

### 🔧 Your turn 1

Change `LANGUAGE` to Hausa and re-run the damage count.

Hausa loses ƙ, ɗ, ɓ and ƴ. Look up what *ƙasa* and *kasa* mean separately — the
hooked letter is a different consonant, not a typographic flourish.

## Normalisation: the decision you must make explicitly

Unicode can represent ọ́ as one character or as three (o + dot below + acute).
Two files that look identical can compare as different strings. `unicodedata`
gives you the two standard forms:

- **NFC** — compose: use the single-character form. Use this for storage and
  comparison.
- **NFD** — decompose: separate base letter from marks. Useful only when you
  *deliberately* want to strip marks and can say why.

In [ ]:
probe = "ọ́"
for form in ("NFC", "NFD"):
    norm = unicodedata.normalize(form, probe)
    print(f"{form}: {len(norm)} code points — "
          + ", ".join(unicodedata.name(c, '?') for c in norm))
print()

nfc = sum(1 for t in df[df["language"] == "Yoruba"]["text"].head(200)
          if str(t) == unicodedata.normalize("NFC", str(t)))
print(f"{nfc} of 200 Yoruba documents are already in NFC form")
print()
print("Normalise once, on load, to NFC. Doing it later means half your counts")
print("were computed on one form and half on the other.")

In [ ]:
# The safe pipeline, in full. Three lines, and none of them is [a-z].
def prepare(text):
    text = unicodedata.normalize("NFC", str(text))
    return re.findall(r"\w+", text.lower())

sanity = "Àwọn ọmọ ìlú — ƙungiyar ɗalibai ta yi taro"
print("input :", sanity)
print("naive :", re.findall(r"[a-z]+", sanity.lower()))
print("safe  :", prepare(sanity))

### 🔧 Your turn 2

Run `prepare()` over one language and compare its vocabulary size with the naive
version's.

The naive tokeniser usually produces *more* types, not fewer — because it splits
one word into fragments at every accented letter. A larger vocabulary looks like
richer language and is in fact destroyed data.

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

**Your turn 1.** Hausa loses its four hooked letters — roughly 4,700 ƙ, 3,500 ɗ,
1,200 ɓ and 500 ƴ in this corpus — and the loss is meaning-bearing: *ƙasa*
(country, land) and *kasa* (below, to fail) are different words that the naive
clean makes identical. Any frequency count, topic model or
classifier built on that text is working with merged categories.

**Your turn 2.** Yes — the naive pattern inflates the type count, because
`àwọn` becomes `w` + `n` and every accented word shatters into fragments. This is
the trap's worst property: the corpus appears to have *more* vocabulary
afterwards, so nothing looks wrong. Only comparing against a Unicode-aware
tokeniser reveals it.

**What to take away.** Three rules, and they cover every language in this pack:

1. Normalise to NFC on load, once.
2. Tokenise with `\w+`, never `[a-z]+`.
3. If you strip diacritics deliberately, say so in your methods and say why —
   it is sometimes defensible (matching against a source that lacks them) and
   never defensible silently.

</details>

## Where this stops

Your text is prepared without losing the languages it is written in. Everything
from here on — comparison, clustering, topic models, stylometry — depends on
having done this correctly.

Next: **Module 04 — comparing corpora**.